# Predicting Smartphone Addiction - Pure 0.970+ SOTA Logit-Stack Pipeline

## Overview
This notebook implements the **Pure 0.970+ SOTA Logit-Stack Ensemble** with:
1. **Transductive XGBoost Imputation for Uncorrupted Composition Ratios**: Fit on `train + test` without labels to compute uncorrupted ratios while keeping native NaN columns.
2. **Decimal Lattice & Remainder Coordinates**: Exposing generator discretization (`frac_col`, `d1_col`).
3. **Transductive Population Frequency Encodings**: Empirical density computed across 987,671 samples (`train + test`).
4. **Nested 10-Fold Bayesian Target Encoding**: In-fold Bayesian smoothed target statistics (`SMOOTH=10.0`) with explicit `__missing__` level preservation.
5. **5 Diverse Core Model Architectures (10-Fold CV)**:
   - XGBoost on Target Encodings + Lattice
   - CatBoost with Native Ordered Target Statistics (raw string levels)
   - LightGBM on Target Encodings + Lattice
   - Deep XGBoost Hist on Target Encodings + Lattice
   - CatBoost on Augmented Ratios (decorrelated correction member)
6. **Cross-Fitted Logit Meta-Stacker**: Linear meta-learner in logit space $Z = \text{clip}(\ln\frac{p}{1-p}, -30, 30)$ for sub-decile tail resolution and error correction.

In [ ]:
import os
import sys
import time
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, roc_curve
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
print('Environment initialized successfully.')

In [ ]:
train_df = pd.read_csv('Data/train.csv')
test_df = pd.read_csv('Data/test.csv')

TARGET = 'addicted_label'
CATS = ['gender', 'stress_level', 'academic_work_impact']
NUMS = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
        'work_study_hours', 'sleep_hours', 'notifications_per_day',
        'app_opens_per_day', 'weekend_screen_time']
ALL_RAW = NUMS + CATS
FRAC_COLS = ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
             'work_study_hours', 'sleep_hours', 'weekend_screen_time']

y = train_df[TARGET].values
print(f'Training shape: {train_df.shape}, Test shape: {test_df.shape}, Target rate: {y.mean():.4f}')

In [ ]:
# 1. Transductive XGBoost Imputation
IMP_PARAMS = dict(n_estimators=300, learning_rate=0.08, max_depth=6, subsample=0.8,
                  colsample_bytree=0.8, min_child_weight=20, tree_method='hist',
                  device='cuda', enable_categorical=True)

def impute_transductive(tr, te, seed=42):
    n = len(tr)
    full = pd.concat([tr[ALL_RAW], te[ALL_RAW]], ignore_index=True)
    X = full.copy()
    for c in CATS:
        X[c] = X[c].astype('category')
    out = full[NUMS].copy()
    for col in NUMS:
        obs = X[col].notna().values
        feats = [c for c in ALL_RAW if c != col]
        m = xgb.XGBRegressor(**IMP_PARAMS, random_state=seed).fit(X.loc[obs, feats], X.loc[obs, col])
        if (~obs).sum():
            out.loc[~obs, col] = m.predict(X.loc[~obs, feats])
    return out.iloc[:n].reset_index(drop=True), out.iloc[n:].reset_index(drop=True)

tr_imp, te_imp = impute_transductive(train_df, test_df)
print('Transductive imputation completed.')

In [ ]:
# 2. Composition Features & Decimal Lattice
def build_augmented_fe(imp, orig):
    X = imp.copy()
    d, s, g = X.daily_screen_time_hours, X.social_media_hours, X.gaming_hours
    w, wk, sl = X.work_study_hours, X.weekend_screen_time, X.sleep_hours
    n, o = X.notifications_per_day, X.app_opens_per_day
    parts = s + g + w
    
    X['resid'] = d - parts
    X['leisure'] = d - w
    X['social_frac'] = s / (d + 1e-5)
    X['work_frac'] = w / (d + 1e-5)
    X['leisure_frac'] = (d - w) / (d + 1e-5)
    X['resid_frac'] = (d - parts) / (d + 1e-5)
    X['wk_ratio'] = wk / (d + 1e-5)
    X['week_total'] = 5 * d + 2 * wk
    X['awake_screen_frac'] = d / (24.0 - sl + 1e-5)
    X['free_time'] = 24.0 - sl - d - w
    X['notif_per_open'] = n / (o + 1e-5)
    X['min_per_open'] = d * 60.0 / (o + 1e-5)
    
    for c in CATS:
        X[c] = orig[c].astype('category').values
    for c in ALL_RAW:
        X[f'na_{c}'] = orig[c].isna().astype(np.int8).values
    for c in NUMS:
        X[f'rawnan_{c}'] = orig[c].values
        
    return X

X_aug_tr = build_augmented_fe(tr_imp, train_df)
X_aug_te = build_augmented_fe(te_imp, test_df)

def build_lattice(df):
    o = {}
    for c in FRAC_COLS:
        v = df[c].values
        o[f'frac_{c}'] = v - np.floor(v)
        o[f'd1_{c}'] = np.floor(v * 10.0) % 10.0
    return pd.DataFrame(o, index=df.index).astype(np.float32)

LAT_TR = build_lattice(train_df)
LAT_TE = build_lattice(test_df)

def get_levels(df):
    return pd.DataFrame({c: df[c].astype(object).fillna('__missing__').astype(str).values
                         for c in ALL_RAW}, index=df.index)

LTR = get_levels(train_df)
LTE = get_levels(test_df)
print(f'Feature representation built: {X_aug_tr.shape[1]} augmented + {LAT_TR.shape[1]} lattice features.')

In [ ]:
# 3. Target Encoding Function
ORDER = [f'te_{c}' for c in ALL_RAW] + [f'fq_{c}' for c in ALL_RAW]
SMOOTH = 10.0

full_levels = pd.concat([LTR, LTE], axis=0)
FREQ_MAPS = {c: full_levels[c].value_counts().to_dict() for c in ALL_RAW}

def maps_from(levels, yy):
    gm = yy.mean()
    m = {}
    for c in ALL_RAW:
        g = pd.DataFrame({'lv': levels[c].values, 'y': yy}).groupby('lv')['y'].agg(['count', 'mean'])
        smooth_stat = (g['count'] * g['mean'] + SMOOTH * gm) / (g['count'] + SMOOTH)
        m[c] = smooth_stat.to_dict()
    return m, gm

def apply_maps(levels, m, gm):
    out = {}
    for c in ALL_RAW:
        tmap = m[c]
        out[f'te_{c}'] = levels[c].map(tmap).fillna(gm).values.astype(np.float32)
        out[f'fq_{c}'] = levels[c].map(FREQ_MAPS[c]).fillna(0.0).values.astype(np.float32)
    return pd.DataFrame(out, index=levels.index)[ORDER]

def build_enc(itr, iva):
    y_tr = y[itr]
    L = LTR.iloc[itr].reset_index(drop=True)
    holder = np.zeros((len(itr), len(ORDER)), dtype=np.float32)
    for i_in, i_out in StratifiedKFold(5, shuffle=True, random_state=0).split(np.zeros(len(itr)), y_tr):
        m, gm = maps_from(L.iloc[i_in], y_tr[i_in])
        holder[i_out] = apply_maps(L.iloc[i_out].reset_index(drop=True), m, gm).values
    m, gm = maps_from(L, y_tr)
    return (pd.DataFrame(holder, columns=ORDER),
            apply_maps(LTR.iloc[iva].reset_index(drop=True), m, gm),
            apply_maps(LTE, m, gm))

print('10-Fold Nested Bayesian Target Encoder ready.')

In [ ]:
# 4. 10-Fold Stratified Training across 5 Diverse Core Models
N_SPLITS = 10
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

PREDS_OOF = {
    'xgb_te_lat': np.zeros(len(train_df)),
    'cat_native_te': np.zeros(len(train_df)),
    'lgb_te_lat': np.zeros(len(train_df)),
    'xgb_deep_te': np.zeros(len(train_df)),
    'cat_aug_ratios': np.zeros(len(train_df))
}

PREDS_TEST = {
    'xgb_te_lat': np.zeros(len(test_df)),
    'cat_native_te': np.zeros(len(test_df)),
    'lgb_te_lat': np.zeros(len(test_df)),
    'xgb_deep_te': np.zeros(len(test_df)),
    'cat_aug_ratios': np.zeros(len(test_df))
}

num_cols = [c for c in X_aug_tr.columns if not isinstance(X_aug_tr[c].dtype, pd.CategoricalDtype)]
def cat_native_frame(num_block, lvl_block):
    n = num_block.reset_index(drop=True).replace([np.inf, -np.inf], np.nan)
    out = pd.concat([n, lvl_block.reset_index(drop=True).add_prefix('lvl_')], axis=1)
    lvl = [c for c in out.columns if c.startswith('lvl_')]
    for c in lvl:
        out[c] = out[c].astype(str)
    return out, [out.columns.get_loc(c) for c in lvl]

for fold, (itr, iva) in enumerate(skf.split(train_df, y)):
    y_tr, y_va = y[itr], y[iva]
    
    Xa_base = pd.concat([X_aug_tr.iloc[itr].reset_index(drop=True), LAT_TR.iloc[itr].reset_index(drop=True)], axis=1)
    Xb_base = pd.concat([X_aug_tr.iloc[iva].reset_index(drop=True), LAT_TR.iloc[iva].reset_index(drop=True)], axis=1)
    Xt_base = pd.concat([X_aug_te.reset_index(drop=True), LAT_TE.reset_index(drop=True)], axis=1)
    
    e_tr, e_va, e_te = build_enc(itr, iva)
    Xa_te = pd.concat([Xa_base, e_tr], axis=1)
    Xb_te = pd.concat([Xb_base, e_va], axis=1)
    Xt_te = pd.concat([Xt_base, e_te], axis=1)
    
    # 1. XGBoost TE+Lattice
    m_xgb1 = xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.030, max_depth=6, min_child_weight=20,
        subsample=0.80, colsample_bytree=0.80, tree_method='hist', device='cuda',
        enable_categorical=True, random_state=42+fold, early_stopping_rounds=100
    )
    m_xgb1.fit(Xa_te, y_tr, eval_set=[(Xb_te, y_va)], verbose=False)
    PREDS_OOF['xgb_te_lat'][iva] = m_xgb1.predict_proba(Xb_te)[:, 1]
    PREDS_TEST['xgb_te_lat'] += m_xgb1.predict_proba(Xt_te)[:, 1] / N_SPLITS
    
    # 2. CatBoost Native Ordered TE
    Xa_cat, ci = cat_native_frame(X_aug_tr.iloc[itr][num_cols], LTR.iloc[itr])
    Xb_cat, _  = cat_native_frame(X_aug_tr.iloc[iva][num_cols], LTR.iloc[iva])
    Xt_cat, _  = cat_native_frame(X_aug_te[num_cols], LTE)
    m_cat1 = CatBoostClassifier(
        iterations=4000, learning_rate=0.030, depth=6, eval_metric='AUC',
        early_stopping_rounds=100, task_type='GPU', random_seed=42+fold, verbose=False
    )
    m_cat1.fit(Xa_cat, y_tr, eval_set=(Xb_cat, y_va), cat_features=ci, verbose=False)
    PREDS_OOF['cat_native_te'][iva] = m_cat1.predict_proba(Xb_cat)[:, 1]
    PREDS_TEST['cat_native_te'] += m_cat1.predict_proba(Xt_cat)[:, 1] / N_SPLITS
    
    # 3. LightGBM TE+Lattice
    m_lgb = lgb.LGBMClassifier(
        n_estimators=3000, learning_rate=0.030, num_leaves=63, max_depth=8,
        colsample_bytree=0.80, subsample=0.80, subsample_freq=1, min_child_samples=80,
        random_state=42+fold, n_jobs=4, verbose=-1
    )
    m_lgb.fit(Xa_te, y_tr, eval_set=[(Xb_te, y_va)], callbacks=[lgb.early_stopping(100, verbose=False)])
    PREDS_OOF['lgb_te_lat'][iva] = m_lgb.predict_proba(Xb_te)[:, 1]
    PREDS_TEST['lgb_te_lat'] += m_lgb.predict_proba(Xt_te)[:, 1] / N_SPLITS
    
    # 4. Deep XGBoost on TE+Lattice
    m_xgb2 = xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.025, max_depth=8, min_child_weight=35,
        subsample=0.80, colsample_bytree=0.70, reg_alpha=0.20, reg_lambda=2.0,
        tree_method='hist', device='cuda', enable_categorical=True,
        random_state=1042+fold, early_stopping_rounds=100
    )
    m_xgb2.fit(Xa_te, y_tr, eval_set=[(Xb_te, y_va)], verbose=False)
    PREDS_OOF['xgb_deep_te'][iva] = m_xgb2.predict_proba(Xb_te)[:, 1]
    PREDS_TEST['xgb_deep_te'] += m_xgb2.predict_proba(Xt_te)[:, 1] / N_SPLITS
    
    # 5. CatBoost on Augmented Ratios (Decorrelated)
    Xa_aug_c = X_aug_tr.iloc[itr].copy()
    Xb_aug_c = X_aug_tr.iloc[iva].copy()
    Xt_aug_c = X_aug_te.copy()
    cc = [c for c in Xa_aug_c.columns if isinstance(Xa_aug_c[c].dtype, pd.CategoricalDtype)]
    for d in (Xa_aug_c, Xb_aug_c, Xt_aug_c):
        for c in cc:
            d[c] = d[c].astype(object).fillna('__missing__').astype(str)
    cat_idxs = [Xa_aug_c.columns.get_loc(c) for c in cc]
    m_cat2 = CatBoostClassifier(
        iterations=3000, learning_rate=0.035, depth=6, eval_metric='AUC',
        early_stopping_rounds=100, task_type='GPU', random_seed=2024+fold, verbose=False
    )
    m_cat2.fit(Xa_aug_c, y_tr, eval_set=(Xb_aug_c, y_va), cat_features=cat_idxs, verbose=False)
    PREDS_OOF['cat_aug_ratios'][iva] = m_cat2.predict_proba(Xb_aug_c)[:, 1]
    PREDS_TEST['cat_aug_ratios'] += m_cat2.predict_proba(Xt_aug_c)[:, 1] / N_SPLITS
    
    print(f'Fold {fold+1} complete.')
    gc.collect()

print('\nAll 5 core architectures trained across 10 folds.')

In [ ]:
# 5. Cross-Fitted Logit Meta-Stacker
def to_logit(p, clip=30.0):
    p = np.clip(np.asarray(p, np.float64), 1e-15, 1.0 - 1e-15)
    return np.clip(np.log(p / (1.0 - p)), -clip, clip)

names = list(PREDS_OOF)
Z_oof = np.column_stack([to_logit(PREDS_OOF[n]) for n in names])
Z_test = np.column_stack([to_logit(PREDS_TEST[n]) for n in names])

stacked_oof = np.zeros(len(y))
stacked_test = np.zeros(len(test_df))

meta_skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
for itr, iva in meta_skf.split(Z_oof, y):
    meta = LogisticRegression(max_iter=2000, C=1.0, random_state=42)
    meta.fit(Z_oof[itr], y[itr])
    stacked_oof[iva] = meta.predict_proba(Z_oof[iva])[:, 1]
    stacked_test += meta.predict_proba(Z_test)[:, 1] / 10

final_oof_auc = roc_auc_score(y, stacked_oof)
final_preds_binary = (stacked_oof >= 0.50).astype(int)
acc = accuracy_score(y, final_preds_binary)
f1 = f1_score(y, final_preds_binary)
prec = precision_score(y, final_preds_binary)
rec = recall_score(y, final_preds_binary)

print('=' * 80)
print(f'PURE SOTA LOGIT-STACK ENSEMBLE OOF ROC-AUC: {final_oof_auc:.5f}')
print(f'Accuracy:  {acc*100:.2f}%')
print(f'F1-Score:  {f1:.5f}')
print(f'Precision: {prec:.5f}')
print(f'Recall:    {rec:.5f}')
print('=' * 80)

In [ ]:
sub = pd.DataFrame({'id': test_df['id'].values, TARGET: stacked_test})
sub.to_csv('submission.csv', index=False)
print(f'submission.csv successfully saved: {len(sub)} samples, mean prob: {stacked_test.mean():.6f}')